# Intelligent Crime Detective Platform - Person B Analytics Notebook

**Development Phase:** Day 1 - Development Foundation & Data Inspection Framework  
**Role:** Person B (Data Engineering, Spatial Profiling, Feature Engineering & Classification Models)  

### Scope & Governance Constraints for Day 1
- No ML model training, clustering, PCA, LWR, or classification is executed.
- No synthetic records or invented schemas are fabricated.
- Datasets are inspected strictly from their source schemas once placed in `data/raw/`.
- This notebook provides the standardized environment, path configuration, dataset inspection, and quality assessment pipeline for all subsequent analytics days.

## 1. Environment Setup

Verify runtime environment, Python version, and core library availability.

In [ ]:
import sys
import os
import platform
from pathlib import Path

print(f"Python Version: {platform.python_version()}")
print(f"Platform: {platform.platform()}")

# Verify foundational libraries
try:
    import pandas as pd
    print(f"pandas: {pd.__version__}")
except ImportError as e:
    print(f"pandas not available: {e}")

try:
    import numpy as np
    print(f"numpy: {np.__version__}")
except ImportError as e:
    print(f"numpy not available: {e}")

try:
    import sklearn
    print(f"scikit-learn: {sklearn.__version__}")
except ImportError as e:
    print(f"scikit-learn not available: {e}")

## 2. Project Path Configuration

Configure project root path and import standard configuration utilities from `src.config`.

In [ ]:
# Ensure project root is on sys.path for robust module resolution
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    MODELS_DIR,
    OUTPUTS_DIR,
    ASSETS_DIR,
    EXPECTED_DATASET_NAMES,
    get_project_paths,
    ensure_directories,
)
from src.data_inspection import inspect_dataset, format_inspection_report, print_inspection_report

ensure_directories()

print("Project Paths Configured:")
for name, p in get_project_paths().items():
    print(f"  {name:<20}: {p}")

## 3. Dataset Loading

Check availability of expected datasets in `data/raw/` without assuming schemas or fabricating data.

In [ ]:
print(f"Checking for raw datasets in: {RAW_DATA_DIR}\n")

available_datasets = []
missing_datasets = []

for filename in EXPECTED_DATASET_NAMES:
    file_path = RAW_DATA_DIR / filename
    if file_path.exists() and file_path.is_file():
        available_datasets.append(file_path)
        print(f"[FOUND]     {filename} ({file_path.stat().st_size:,} bytes)")
    else:
        missing_datasets.append(filename)
        print(f"[NOT FOUND] {filename} (Awaiting placement into data/raw/)")

print(f"\nSummary: {len(available_datasets)} available, {len(missing_datasets)} pending ingestion.")

## 4. Dataset Inspection

Inspect the schema, dimensions, and sample records of any available datasets using `src.data_inspection.inspect_dataset`.

In [ ]:
inspection_results = {}

if not available_datasets:
    print("No datasets currently present in data/raw/.")
    print("Place raw CSV files into data/raw/ to inspect their schemas.")
else:
    for dataset_path in available_datasets:
        result = inspect_dataset(dataset_path)
        inspection_results[dataset_path.name] = result
        print_inspection_report(result)

## 5. Missing Value Inspection

Analyze missing value distribution across features in the ingested datasets.

In [ ]:
if not inspection_results:
    print("Awaiting dataset ingestion for missing value inspection.")
else:
    for name, res in inspection_results.items():
        print(f"=== Missing Values: {name} ===")
        if res['status'] == 'success':
            missing_dict = res['missing_values']
            cols_with_missing = {k: v for k, v in missing_dict.items() if v > 0}
            if cols_with_missing:
                for col, count in cols_with_missing.items():
                    pct = res['missing_percentage'].get(col, 0.0)
                    print(f"  {col}: {count} missing ({pct}%)")
            else:
                print("  No missing values detected across all columns.")
        else:
            print(f"  Cannot inspect: {res['error_message']}")

## 6. Duplicate Inspection

Detect duplicate records to identify data integrity issues prior to feature engineering.

In [ ]:
if not inspection_results:
    print("Awaiting dataset ingestion for duplicate record inspection.")
else:
    for name, res in inspection_results.items():
        print(f"=== Duplicate Analysis: {name} ===")
        if res['status'] == 'success':
            print(f"  Duplicate Rows: {res['duplicate_count']:,} ({res['duplicate_percentage']}%) of {res['num_rows']:,} rows")
        else:
            print(f"  Cannot inspect: {res['error_message']}")

## 7. Preliminary Data Quality Report

Synthesizes structural observations, encoding stability, and completeness for available datasets.

In [ ]:
if not inspection_results:
    print("Preliminary Data Quality Summary: [PENDING DATASET INGESTION]")
    print("Once datasets are provided, this section summarizes schema health, dtypes, and data anomalies.")
else:
    summary_rows = []
    for name, res in inspection_results.items():
        if res['status'] == 'success':
            summary_rows.append({
                "Dataset": name,
                "Rows": res['num_rows'],
                "Columns": res['num_columns'],
                "Encoding": res['encoding_used'],
                "Missing Cells": res['total_missing'],
                "Duplicates": res['duplicate_count'],
            })
        else:
            summary_rows.append({
                "Dataset": name,
                "Rows": None,
                "Columns": None,
                "Encoding": None,
                "Missing Cells": None,
                "Duplicates": None,
            })
    
    if 'pd' in globals():
        summary_df = pd.DataFrame(summary_rows)
        print(summary_df.to_string(index=False))
    else:
        print(summary_rows)

## 8. Dataset Inventory

Tracking expected datasets, analytical roles, and ingestion status for Day 2 processing.

In [ ]:
inventory = [
    {
        "Dataset": "homicide-data.csv",
        "Target Domain": "Homicide Incident Analysis & Solvability Estimation",
        "Planned Person B Modules": "Feature Engineering, PCA, Naive Bayes / ID3 Solvability Classification, k-NN",
        "Status": "Ready for Inspection" if (RAW_DATA_DIR / "homicide-data.csv").exists() else "Awaiting File"
    },
    {
        "Dataset": "dstrIPC_1_2014.csv",
        "Target Domain": "District-Level IPC Crime Statistics",
        "Planned Person B Modules": "Geographic Profiling, Statistical Aggregation, Comparative Analytics",
        "Status": "Ready for Inspection" if (RAW_DATA_DIR / "dstrIPC_1_2014.csv").exists() else "Awaiting File"
    },
    {
        "Dataset": "TN-murder-2023.csv",
        "Target Domain": "Tamil Nadu 2023 Crime Analytics",
        "Planned Person B Modules": "Tamil Nadu Specific Profiling, Folium Spatial Mapping, LWR",
        "Status": "Ready for Inspection" if (RAW_DATA_DIR / "TN-murder-2023.csv").exists() else "Awaiting File"
    },
    {
        "Dataset": "TN-2020-2022-total.csv",
        "Target Domain": "Tamil Nadu Multi-Year Trend Analysis (2020-2022)",
        "Planned Person B Modules": "Temporal Trend Modeling, District Comparison, Streamlit Visualization",
        "Status": "Ready for Inspection" if (RAW_DATA_DIR / "TN-2020-2022-total.csv").exists() else "Awaiting File"
    },
]

print("=== DATASET INVENTORY (DAY 1) ===")
for item in inventory:
    print(f"- {item['Dataset']}: [{item['Status']}]")
    print(f"    Target Domain: {item['TargetDomain'] if 'TargetDomain' in item else item['Target Domain']}")
    print(f"    Person B Modules: {item['Planned Person B Modules']}")